In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train = pd.read_csv("/kaggle/input/titanic-extended/train.csv")
test = pd.read_csv("/kaggle/input/titanic-extended/test.csv")

train.head()

In [ ]:
null_vals = train.isna().sum().sort_values(ascending=False)
print("Train:")
print(null_vals[null_vals > 0 ])

print("------")

null_vals = test.isna().sum().sort_values(ascending=False)
print("Test:")
print(null_vals[null_vals > 0 ])

In [ ]:
ntrain = train.shape[0]
ntest = test.shape[0]

y_train = train['Survived'].values
test_IDs = test['PassengerId'].copy()

all_data = pd.concat((train, test)).reset_index(drop=True)
all_data.drop(['Survived','PassengerId','Cabin','Body', 'Ticket','Age_wiki','Name_wiki','WikiId','Hometown','Destination'], axis=1, inplace=True)
all_data

In [ ]:
null_vals = all_data.isna().sum().sort_values(ascending=False)
null_vals[null_vals > 0 ]

all_data['Lifeboat'] = all_data['Lifeboat'].fillna('None')

for col in null_vals[(null_vals < 10) & ( null_vals > 0)].index : 
    all_data[col] = all_data.groupby(["Pclass","Sex"])[col].transform(
        lambda x: x.fillna(x.mode()[0]))
all_data.isna().sum().sort_values(ascending=False)

In [ ]:
all_data['FamilySize'] = all_data['SibSp'] + all_data['Parch'] + 1
all_data['IsAlone'] = all_data['FamilySize'] <= 1

In [ ]:
all_data['Title'] = all_data['Name'].apply(lambda x : x.split(' ')[1].strip('123,./!?'))
all_data.drop('Name', axis = 1 , inplace = True)

all_data['Title'] = all_data['Title'].apply(lambda x : x if x in ['Mr','Miss','Mrs','Master'] else 'NoTitle')
all_data['Title'].value_counts()

In [ ]:
all_data['Age'] = all_data.groupby(['Pclass', 'Sex','Title']).Age.apply(lambda x: x.fillna(x.median()))
all_data["Age"].value_counts()

In [ ]:
categorical_col = []
categorical_col = []
for col in all_data.columns:
    if all_data[col].dtype in [object, bool] and len(all_data[col].unique()) <= 50:
        if all_data[col].dtype in [object, bool] and len(all_data[col].unique()) <= 50:
            categorical_col.append(col)
categorical_col

In [ ]:
for col in categorical_col:
    all_data[col] = all_data[col].astype("category").cat.codes
all_data[categorical_col].head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(categories='auto')

array_hot_encoded = ohe.fit_transform(all_data[categorical_col]).toarray()

data_hot_encoded = pd.DataFrame(array_hot_encoded, index=all_data.index)

In [ ]:
data_other_cols = all_data.drop(columns=categorical_col)

all_data = pd.concat([data_hot_encoded, data_other_cols], axis=1)
all_data.head()

In [ ]:
all_data.columns = all_data.columns.astype(str)

In [ ]:
train_data = all_data[:ntrain].copy()
test_data = all_data[ntrain:]

In [ ]:
train_data['Survived'] = y_train
train_data['Survived'] = train_data['Survived'].astype("category").cat.codes

train_data.head()

In [ ]:
k = 40
f,ax = plt.subplots(figsize=(11, 11))
cols = train_data.corr().abs().nlargest(k, 'Survived')['Survived'].index  
cm = np.corrcoef(train_data[cols].values.T)
sns.set(font_scale=1.25)
hm = sns.heatmap(cm, cbar=True, annot=True, square=True, fmt='.2f', annot_kws={'size': 10}, yticklabels=cols.values, xticklabels=cols.values)
plt.show()

train_data = train_data[cols]
test_data = test_data[cols.drop('Survived')]
train_data.shape , test_data.shape

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
clf = LocalOutlierFactor(contamination = 0.02)

outliers = clf.fit_predict(train_data)

train_data_cleaned = train_data[np.where(outliers == 1, True, False)]
train_data_cleaned.shape

In [ ]:
sns.countplot(x = train_data['Survived'], label = "Original", color = 'red' , alpha = 0.5)
sns.countplot(x = train_data_cleaned['Survived'], label = "Modified" , color = 'blue',alpha = 0.5)
X_train = train_data_cleaned.drop('Survived', axis = 1)
y_train = train_data_cleaned.Survived

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE( random_state=42)

print(y_train.value_counts())
X_resampled, y_resampled = smote.fit_resample(X_train,y_train)
print(y_resampled.value_counts())

sns.kdeplot(y_train, label = "Original" , color = 'red')
sns.kdeplot(y_resampled, label = "Modified")
X_train = X_resampled
y_train = y_resampled

In [ ]:
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, Normalizer, PowerTransformer
from sklearn.model_selection import GridSearchCV , cross_val_score

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
test_data_scaled = scaler.transform(test_data)

In [ ]:
from catboost import CatBoostClassifier
pipeline_catB=Pipeline([ ('CatBoostClassifier',CatBoostClassifier(verbose = 0 ))])

cat = CatBoostClassifier(eval_metric='Accuracy',
                        verbose = 0)

#USING GRID SEARCH
params_catB = {
    'learning_rate' : [0.01,0.02,0.03,],
          'depth': [6,7,],
    'iterations' : [450,1000,]
         }

grid_search_cat = GridSearchCV(estimator=cat,verbose = 1, param_grid=params_catB, cv = 3,scoring='accuracy',n_jobs=-1).fit(X_train.values, y_train.values.ravel())
cat_best = grid_search_cat.best_estimator_
print('CatBoost Best Score',grid_search_cat.best_score_)
print('CatBoost Best Parmas',grid_search_cat.best_params_)
print('CatBoost Accuracy',cross_val_score(cat_best,X_train.values, y_train.values.ravel(), cv=3).mean())

In [ ]:
predictions = cat_best.predict(test_data.values)

In [ ]:
sub = pd.DataFrame()
sub['PassengerId'] = test_IDs
sub['Survived'] = predictions
sub['Survived'] = sub['Survived']
sub.to_csv('submission.csv',index=False)